# Lab | BabyAGI with agent

**Change the planner objective below by changing the objective and the associated prompts and potential tools and agents - Wear your creativity and AI engineering hats
You can't get this wrong!**

You would need the OpenAI API KEY and the [SerpAPI KEY](https://serpapi.com/manage-api-keyhttps://serpapi.com/manage-api-key) to run this lab.


## BabyAGI with Tools

This notebook builds on top of [baby agi](baby_agi.html), but shows how you can swap out the execution chain. The previous execution chain was just an LLM which made stuff up. By swapping it out with an agent that has access to tools, we can hopefully get real reliable information

## Install and Import Required Modules

In [1]:
!pip install langchain_community langchain_experimental langchain_openai

## Connect to the Vector Store

Depending on what vectorstore you use, this step may look different.

In [2]:
!pip install -q langchain langchain-openai langchain-community faiss-cpu python-dotenv

In [3]:
import os
from typing import List
from dotenv import load_dotenv, find_dotenv

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv(find_dotenv())

print("Imports OK")

Imports OK


/var/folders/3k/my8rx_2936x18t9wy2ywg_lm0000gn/T/ipykernel_7729/2309429088.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [5]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

response = llm.invoke("Explain BabyAGI in one simple sentence.")
print(response.content)

BabyAGI is a simplified artificial intelligence framework designed to autonomously set and achieve goals by iteratively planning and executing tasks.


In [6]:
prompt = PromptTemplate.from_template(
    "Answer this question clearly: {question}"
)

chain = prompt | llm | StrOutputParser()

result = chain.invoke({
    "question": "What is BabyAGI?"
})

print(result)

BabyAGI is a simplified framework or concept for developing artificial general intelligence (AGI) systems. It typically refers to a model or approach that aims to mimic certain aspects of human cognitive abilities, such as learning, reasoning, and problem-solving, in a more manageable and less complex manner than full AGI. The term "BabyAGI" suggests a nascent or early-stage version of AGI, focusing on foundational capabilities that can be built upon as technology and understanding of intelligence evolve. It may involve using machine learning techniques to create systems that can adapt and improve over time, but it does not represent a fully realized AGI.


In [7]:
memory_store = [
    "Initial memory"
]

def add_memory(text):
    memory_store.append(text)

def get_context(query, k=3):
    return "\n".join(memory_store[-k:])

In [8]:
print(get_context("memory"))

Initial memory


## Define the Chains

BabyAGI relies on three LLM chains:
- Task creation chain to select new tasks to add to the list
- Task prioritization chain to re-prioritize tasks
- Execution Chain to execute the tasks


NOTE: in this notebook, the Execution chain will now be an agent.

In [9]:
task_creation_prompt = PromptTemplate.from_template("""
You are a task creation AI.

Objective:
{objective}

Current task result:
{result}

Incomplete tasks:
{task_description}

Create new tasks that help complete the objective.
Return them as a numbered list.
""")

task_creation_chain = task_creation_prompt | llm | StrOutputParser()


task_prioritization_prompt = PromptTemplate.from_template("""
You are a task prioritization AI.

Objective:
{objective}

Tasks:
{task_names}

Reorder the tasks by importance.
Return them as a numbered list.
""")

task_prioritization_chain = task_prioritization_prompt | llm | StrOutputParser()


execution_prompt = PromptTemplate.from_template("""
You are an AI agent executing one task.

Objective:
{objective}

Current task:
{task}

Relevant context:
{context}

Return a useful, direct result.
""")

execution_chain = execution_prompt | llm | StrOutputParser()

print("Chains OK")

Chains OK


### Run the BabyAGI

Now it's time to create the BabyAGI controller and watch it try to accomplish your objective.

In [11]:
from collections import deque

In [12]:
OBJECTIVE = "Create a 7-day study plan to learn LangChain agents"

task_list = deque([
    {
        "task_id": 1,
        "task_name": "Create a high-level structure for the 7-day study plan"
    }
])

next_task_id = 2

print("Objective:", OBJECTIVE)
print("Task list:", list(task_list))

Objective: Create a 7-day study plan to learn LangChain agents
Task list: [{'task_id': 1, 'task_name': 'Create a high-level structure for the 7-day study plan'}]


In [13]:
max_iterations = 3

for iteration in range(max_iterations):
    print("\n" + "="*80)
    print(f"ITERATION {iteration + 1}")
    print("="*80)

    if not task_list:
        print("Task list empty. Stopping.")
        break

    current_task = task_list.popleft()

    print("\nCurrent task:")
    print(current_task)

    context = get_context(OBJECTIVE)

    result = execution_chain.invoke({
        "objective": OBJECTIVE,
        "task": current_task["task_name"],
        "context": context
    })

    print("\nResult:")
    print(result)

    add_memory(
        f"Task: {current_task['task_name']}\nResult: {result}"
    )

    incomplete_tasks = ", ".join([t["task_name"] for t in task_list]) or "None"

    new_tasks_raw = task_creation_chain.invoke({
        "objective": OBJECTIVE,
        "result": result,
        "task_description": incomplete_tasks
    })

    print("\nNew tasks raw:")
    print(new_tasks_raw)

    new_tasks = []

    for line in new_tasks_raw.split("\n"):
        line = line.strip()

        if line and (line[0].isdigit() or line.startswith("-")):
            clean_task = line.lstrip("0123456789.-) ").strip()

            if clean_task:
                new_tasks.append({
                    "task_id": next_task_id,
                    "task_name": clean_task
                })
                next_task_id += 1

    task_list.extend(new_tasks)

    task_names = "\n".join([t["task_name"] for t in task_list])

    prioritized_raw = task_prioritization_chain.invoke({
        "objective": OBJECTIVE,
        "task_names": task_names
    })

    print("\nPrioritized tasks raw:")
    print(prioritized_raw)

    print("\nCurrent task list:")
    for task in task_list:
        print(task)


ITERATION 1

Current task:
{'task_id': 1, 'task_name': 'Create a high-level structure for the 7-day study plan'}

Result:
### 7-Day Study Plan to Learn LangChain Agents

#### Day 1: Introduction to LangChain
- **Objective**: Understand the basics of LangChain and its purpose.
- **Activities**:
  - Read the official LangChain documentation.
  - Watch introductory videos/tutorials on LangChain.
  - Explore use cases and applications of LangChain agents.

#### Day 2: Setting Up the Environment
- **Objective**: Prepare your development environment for LangChain.
- **Activities**:
  - Install Python and necessary libraries (LangChain, OpenAI, etc.).
  - Set up a virtual environment.
  - Run a simple LangChain example to ensure everything is working.

#### Day 3: Understanding LangChain Components
- **Objective**: Learn about the core components of LangChain.
- **Activities**:
  - Study the architecture of LangChain (chains, agents, tools).
  - Read about different types of agents and their

In [14]:
def calculator_tool(expression):
    return str(eval(expression))

tool_result = calculator_tool("25 * 7")
print(tool_result)

175


## Now applying SERPAPI for an example of weather

In [16]:
!pip install -q google-search-results

In [17]:
from langchain_community.utilities import SerpAPIWrapper

search = SerpAPIWrapper()

def search_tool(query):
    return search.run(query)

print(search_tool("San Francisco weather today"))

{'type': 'weather_result', 'temperature': '66', 'unit': 'Fahrenheit', 'precipitation': '0%', 'humidity': '62%', 'wind': '11 mph', 'location': 'San Francisco, CA', 'date': 'Monday 12:00 PM', 'weather': 'Mostly sunny'}


In [18]:
OBJECTIVE = "Write a weather report for San Francisco today"

task_list = deque([
    {
        "task_id": 1,
        "task_name": "Search the current weather in San Francisco today"
    }
])

next_task_id = 2
memory_store = ["Initial memory"]

In [19]:
execution_prompt = PromptTemplate.from_template("""
You are an AI agent executing one task.

Objective:
{objective}

Current task:
{task}

Relevant context:
{context}

Search result:
{search_result}

Use the search result if it is relevant.
Return a useful, direct result.
""")

execution_chain = execution_prompt | llm | StrOutputParser()

In [20]:
search_result = search_tool(current_task["task_name"])

In [21]:
result = execution_chain.invoke({
    "objective": OBJECTIVE,
    "task": current_task["task_name"],
    "context": context,
    "search_result": search_result
})

In [24]:
from collections import deque

OBJECTIVE = "Write a weather report for San Francisco today"

task_list = deque([
    {
        "task_id": 1,
        "task_name": "Search the current weather in San Francisco today"
    }
])

next_task_id = 2
memory_store = ["Initial memory"]

print(task_list)

deque([{'task_id': 1, 'task_name': 'Search the current weather in San Francisco today'}])


In [25]:
max_iterations = 2

for iteration in range(max_iterations):
    print("\n" + "="*80)
    print(f"ITERATION {iteration + 1}")
    print("="*80)

    if not task_list:
        print("Task list empty. Stopping.")
        break

    current_task = task_list.popleft()

    print("\nCurrent task:")
    print(current_task)

    context = get_context(OBJECTIVE)

    search_result = search_tool(current_task["task_name"])

    result = execution_chain.invoke({
       "objective": OBJECTIVE,
       "task": current_task["task_name"],
       "context": context,
       "search_result": search_result
    })

    print("\nResult:")
    print(result)

    add_memory(
        f"Task: {current_task['task_name']}\nResult: {result}"
    )

    incomplete_tasks = ", ".join([t["task_name"] for t in task_list]) or "None"

    new_tasks_raw = task_creation_chain.invoke({
        "objective": OBJECTIVE,
        "result": result,
        "task_description": incomplete_tasks
    })

    print("\nNew tasks raw:")
    print(new_tasks_raw)

    new_tasks = []

    for line in new_tasks_raw.split("\n"):
        line = line.strip()

        if line and (line[0].isdigit() or line.startswith("-")):
            clean_task = line.lstrip("0123456789.-) ").strip()

            if clean_task:
                new_tasks.append({
                    "task_id": next_task_id,
                    "task_name": clean_task
                })
                next_task_id += 1

    task_list.extend(new_tasks)

    task_names = "\n".join([t["task_name"] for t in task_list])

    prioritized_raw = task_prioritization_chain.invoke({
        "objective": OBJECTIVE,
        "task_names": task_names
    })

    print("\nPrioritized tasks raw:")
    print(prioritized_raw)

    print("\nCurrent task list:")
    for task in task_list:
        print(task)


ITERATION 1

Current task:
{'task_id': 1, 'task_name': 'Search the current weather in San Francisco today'}

Result:
**Weather Report for San Francisco - Monday**

Today in San Francisco, the weather is sunny with a temperature of 65°F. There is no expected precipitation, and humidity levels are at 76%. Winds are blowing at 14 mph. Enjoy the beautiful day!

New tasks raw:
1. Research and include the forecasted high and low temperatures for San Francisco today.
2. Add information about the UV index and any relevant health advisories.
3. Provide a brief overview of the air quality index for the day.
4. Include a summary of the expected cloud cover and visibility conditions.
5. Mention any significant weather events or alerts in the surrounding areas.
6. Suggest appropriate clothing or activities based on the weather conditions.
7. Include a brief historical comparison of today's weather to the same date last year.

Prioritized tasks raw:
Here’s the reordered list of tasks by importance 